In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout
import time


async def get_html(url, selector, sleep=5, retries=5):
    html = None
    for i in range(1, retries + 1):
        time.sleep(sleep * i)
        try:
            async with async_playwright() as p:
                browser = await p.webkit.launch()
                page = await browser.new_page()
                await page.goto(url)
                print(await page.title())
                html = await page.inner_html(selector)
        except PlaywrightTimeout:
            print(f"Timeout error on {url}")
            continue
        else:
            break
    return html

In [ ]:
html = await get_html(
    "https://finance.yahoo.com/markets/stocks/most-active/", ".tableContainer"
)

In [ ]:
soup = BeautifulSoup(html)
stocks = pd.read_html(str(soup))[0]

In [ ]:
stocks = stocks.dropna(axis=0, how="all")
stocks = stocks.dropna(axis=1, how="all")
stocks = stocks.drop("52 Wk Range", axis=1)

In [ ]:
def extract_price(all_prices):
    price = str(all_prices).split(" ")[0]
    return price


def format_numbers(num):
    num = str(num)
    if num.endswith("%"):
        num = num.split("%")[0]
    elif num.endswith("M"):
        num = int(num) * 1e6
    elif num.endswith("B"):
        num = int(num) * 1e9
    elif num.endswith("T"):
        num = int(num) * 1e12


stocks["Price"] = stocks["Price"].apply(extract_price)

In [ ]:
stocks["Change %"] = stocks["Change %"].apply(format_numbers)
stocks["Volume"] = stocks["Volume"].apply(format_numbers)
stocks["Avg Vol (3M)"] = stocks["Avg Vol (3M)"].apply(format_numbers)
stocks["Market Cap"] = stocks["Market Cap"].apply(format_numbers)